# Photon Flux & Ferrioxalate Actinometry

Run the cells top to bottom.

1. **Setup** — imports and shared functions
2. **Part 1** — Photon flux from the calibrated photodiode (voltage after reference)
3. **Part 2** — Irradiation time calculator for ferrioxalate (before experiment)
4. **Part 3** — Photon flux from ferrioxalate actinometry (after experiment)

The photon flux in mol s⁻¹ from Part 1 or Part 3 is the input for `03_QY_calculation.py`.

---
## Setup — Imports & Shared Functions
Run this cell first.

In [1]:
import csv
import os
import numpy as np
import pandas as pd
from scipy import constants

# ---------------------------------------------------------------------------
# Photodiode calibration data and correction factor interpolation
# ---------------------------------------------------------------------------

CALIBRATION_DATA = {
    "wavelength": [
        260,270,280,290,300,310,320,330,340,350,360,370,380,390,400,
        410,420,430,440,450,460,470,480,490,500,510,520,530,540,550,
        560,570,580,590,600,610,620,630,640,650,660
    ],
    "high": [
        10.042,9.892,9.493,8.934,8.301,7.654,7.036,6.467,5.956,5.503,
        5.105,4.758,4.454,4.189,3.958,3.756,3.578,3.423,3.285,3.164,
        3.056,2.960,2.874,2.797,2.727,2.663,2.605,2.552,2.502,2.455,
        2.410,2.367,2.325,2.284,2.244,2.203,2.162,2.121,2.080,2.038,
        1.995
    ],
    "low": [
        0.502,0.495,0.475,0.447,0.415,0.383,0.352,0.323,0.298,0.275,
        0.255,0.238,0.223,0.209,0.198,0.188,0.179,0.171,0.164,0.158,
        0.153,0.148,0.144,0.140,0.136,0.133,0.130,0.128,0.125,0.123,
        0.120,0.118,0.116,0.114,0.112,0.110,0.108,0.106,0.104,0.102,
        0.100
    ]
}

def get_correction_factor(wavelength, channel='low'):
    """Interpolate photodiode correction factor [mW/V]."""
    return np.interp(wavelength, CALIBRATION_DATA['wavelength'], CALIBRATION_DATA[channel])


# ---------------------------------------------------------------------------
# UV-Vis CSV parser
# ---------------------------------------------------------------------------

def produce_UV_Vis_dataframe(file):
    """Read a Cary UV-Vis csv, raw or cleaned by 00_UVVis_csv_changer.

    Layout: row 0 = spectrum names (every 2nd column), row 1 = 'Wavelength (nm),Abs',
    then data as x/y column pairs, optionally followed by an empty line and the
    instrument metadata. Works for spectra and for kinetics files (x = time).
    Returns a DataFrame: column 0 = x of the first spectrum, then one column per spectrum.
    """
    with open(file, 'rb') as f:
        text = f.read().decode('utf-8', errors='replace')
    text = text.replace('\r\r\n', '\n').replace('\r\n', '\n').replace('\r', '\n')

    def is_num(s):
        try:
            float(s)
            return True
        except ValueError:
            return False

    header, data = [], []
    for row in csv.reader(text.split('\n')):
        if not any(c.strip() for c in row):     # empty or ',,,' line
            if data:
                break                            # metadata block starts
            continue
        if row and is_num(row[0].strip()):
            data.append(row)
        elif data:
            break
        else:
            header.append(row)
    if not data:
        raise ValueError(f"No numeric data found in {os.path.basename(file)}")

    n_cols = max(len(r) for r in data)
    arr = np.full((len(data), n_cols), np.nan)
    for i, r in enumerate(data):
        for j, c in enumerate(r):
            if c.strip() and is_num(c.strip()):
                arr[i, j] = float(c)

    y_idx = [j for j in range(1, n_cols, 2) if not np.all(np.isnan(arr[:, j]))]
    names = []
    for j in y_idx:
        name = header[0][j - 1].strip() if header and len(header[0]) > j - 1 else ''
        names.append(name or f'spectrum {len(names) + 1}')
    names = [f'{n} ({i})' if names.count(n) > 1 else n for i, n in enumerate(names)]

    df = pd.DataFrame(arr[:, y_idx], columns=names)
    df.insert(0, 'x', arr[:, 0])
    return df.dropna(subset=['x']).reset_index(drop=True)

def extract_absorption_at_wavelength(dataframe, wavelength=510):
    idx = (dataframe.iloc[:, 0] - wavelength).abs().to_numpy().argmin()
    return dataframe.iloc[idx, 1:].tolist()


# ---------------------------------------------------------------------------
# QYDS .dat file parser
# ---------------------------------------------------------------------------

def produce_QYDS_dataframe(file):
    data = {'Runtime': [], 'Illum Time': [], 'Cell Value': []}
    with open(file, 'r') as f:
        lines = f.readlines()
    for line in lines:
        if line.startswith('#') or line.strip().startswith('Number'):
            continue
        parts = line.split('\t')
        if len(parts) >= 7:
            try:
                data['Runtime'].append(float(parts[3].replace(',', '.')))
                data['Illum Time'].append(float(parts[4].replace(',', '.')))
                data['Cell Value'].append(float(parts[6].replace(',', '.')))
            except ValueError:
                continue
    return pd.DataFrame(data).astype(float)


# ---------------------------------------------------------------------------
# Ferrioxalate actinometer settings: (concentration / mM, quantum yield)
# ---------------------------------------------------------------------------

WAVELENGTH_SETTINGS = {
    405: (6, 1.14), 436: (6, 1.11), 458: (10, 1.12),
    488: (80, 1.08), 514: (200, 0.93)
}

def actinometer_settings(wavelength):
    if 254 <= wavelength <= 366:
        return 6, 1.23
    if wavelength in WAVELENGTH_SETTINGS:
        return WAVELENGTH_SETTINGS[wavelength]
    raise ValueError(f'No actinometry settings defined for {wavelength} nm.')

# Fixed actinometry volumes and constants
V1, V2, V3 = 3, 1, 10    # mL: irradiated, transferred, total analytic volume
eps, d     = 11100, 1    # L mol-1 cm-1 (Fe(phen)3 at 510 nm), cm


print('Setup complete.')

Setup complete.


---
## Part 1 — Photon Flux from Photodiode

Photon flux from the voltage measured after the reference cuvette:
P = U · CF(λ),  q = P · λ / (h c N_A)

For an LED use its central wavelength. Copy the value in mol s⁻¹ into `03_QY_calculation.py`.

In [4]:
# ── Set your parameters here ───────────────────────────────────────────────
wavelength = 365     # nm  (LED central wavelength)
channel    = 'high'  # 'low' or 'high'
voltage    = 9.87     # V   (measured voltage after reference)
volume     = 2       # mL  (irradiated volume, only for the per-volume value)
# ──────────────────────────────────────────────────────────────────────────

cf          = get_correction_factor(wavelength, channel)
power_mW    = voltage * cf
photon_flux_diode = power_mW * 1e-3 * wavelength * 1e-9 / (
    constants.Planck * constants.speed_of_light * constants.Avogadro)   # mol s-1

print('─' * 52)
print(f'  Wavelength              : {wavelength} nm')
print(f'  Channel                 : {channel}')
print(f'  Voltage after reference : {voltage} V')
print(f'  Correction factor       : {cf:.4f} mW V\u207b\u00b9')
print(f'  Optical power           : {power_mW:.3f} mW')
print('─' * 52)
print(f'  Photon flux             : {photon_flux_diode:.3e} mol s\u207b\u00b9')
print(f'                            {photon_flux_diode*constants.Avogadro:.3e} s\u207b\u00b9')
print(f'                            {photon_flux_diode/(volume*1e-3):.3e} mol s\u207b\u00b9 L\u207b\u00b9')
print('─' * 52)

────────────────────────────────────────────────────
  Wavelength              : 365 nm
  Channel                 : high
  Voltage after reference : 9.87 V
  Correction factor       : 4.9315 mW V⁻¹
  Optical power           : 48.674 mW
────────────────────────────────────────────────────
  Photon flux             : 1.485e-07 mol s⁻¹
                            8.944e+16 s⁻¹
                            7.426e-05 mol s⁻¹ L⁻¹
────────────────────────────────────────────────────


---
## Part 2 — Irradiation Time Calculator

Run **before** your experiment to estimate the required irradiation time, using the photon flux from Part 1 and the same wavelength.

In [ ]:
# ── Set your parameters here ───────────────────────────────────────────────
desired_absorbance = 0.5    # target ΔA at 510 nm
# ──────────────────────────────────────────────────────────────────────────

conc, QY    = actinometer_settings(wavelength)
result_time = desired_absorbance * V1 * V3 / (QY * eps * V2 * d * photon_flux_diode) * 1e-3

print('─' * 52)
print(f'  Wavelength              : {wavelength} nm')
print(f'  Photon flux (Part 1)    : {photon_flux_diode:.2e} mol s\u207b\u00b9')
print(f'  Ferrioxalate            : {conc} mM, \u03a6 = {QY}')
print(f'  Desired absorbance      : {desired_absorbance}')
print('─' * 52)
print(f'  Required irradiation    : {result_time:.0f} s  ({result_time/60:.1f} min)')
print('─' * 52)

---
## Part 3 — Photon Flux from Actinometry

Use **after** your experiment. The CSV should contain the blank spectrum (column 1) and the irradiated spectrum (column 2). ΔA at 510 nm is used to back-calculate the photon flux.

**Supported wavelengths and actinometry settings:**

| λ / nm | Concentration / mM | Quantum yield |
|---|---|---|
| 254 – 366 | 6 | 1.23 |
| 405 | 6 | 1.14 |
| 436 | 6 | 1.11 |
| 458 | 10 | 1.12 |
| 488 | 80 | 1.08 |
| 514 | 200 | 0.93 |

Edit the parameters below, then run the cell.

In [ ]:
# ── Set your parameters here ───────────────────────────────────────────────
uv_file          = r"E:\01_REPHORM\03_UV_Vis\260921_CMuG08\01_Raw\260921_Actinometry_UIV.csv" # path to UV-Vis CSV file
wavelength       = 365   # nm  (irradiation wavelength)
irradiation_time = 277   # s
# ──────────────────────────────────────────────────────────────────────────

conc, QY    = actinometer_settings(wavelength)
df          = produce_UV_Vis_dataframe(uv_file)
absorbances = extract_absorption_at_wavelength(df, wavelength=510)
dA          = absorbances[1] - absorbances[0]
photon_flux = dA * V1 * V3 / (QY * eps * V2 * d * irradiation_time) * 1e-3

print('─' * 55)
print('INPUT PARAMETERS')
print(f'  Irradiation wavelength        : {wavelength} nm')
print(f'  Irradiation time              : {irradiation_time} s')
print('USED PARAMETERS')
print(f'  Concentration                 : {conc} mM')
print(f'  Quantum yield                 : {QY}')
print(f'  V\u2081 (irradiated)               : {V1} mL')
print(f'  V\u2082 (transferred)              : {V2} mL')
print(f'  V\u2083 (analytic total)           : {V3} mL')
print(f'  \u03b5 (510 nm)                    : {eps} L mol\u207b\u00b9 cm\u207b\u00b9')
print(f'  Path length                   : {d} cm')
print('─' * 55)
print('RESULTS')
print(f'  A (blank)                     : {absorbances[0]:.4f}')
print(f'  A (irradiated)                : {absorbances[1]:.4f}')
print(f'  \u0394A at 510 nm                  : {dA:.4f}')
print(f'  Photon flux from actinometry  : {photon_flux:.3e} mol s\u207b\u00b9')
if 'photon_flux_diode' in globals():
    print(f'  Photon flux from photodiode   : {photon_flux_diode:.3e} mol s\u207b\u00b9 '
          f'(ratio {photon_flux/photon_flux_diode:.3f})')
print('─' * 55)